# LSTM-arithmetic



In [ ]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
# Generative AI assistance: OpenAI Codex helped complete TODOs and validation.
# Install the template's optional OpenCC dependency only when it is missing.
try:
    import opencc
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencc-python-reimplemented'])
    import opencc
import os
from sklearn.model_selection import train_test_split

# VS Code Colab uses remote files: upload data/ or put HW2/data on Drive.
# Optional overrides: HW2_DATA_DIR (CSVs), HW2_OUTPUT_DIR (persistent artifacts).
from pathlib import Path
import sys
import platform
import importlib.metadata

candidates = [Path(os.environ['HW2_DATA_DIR'])] if 'HW2_DATA_DIR' in os.environ else [
    Path('./data'), Path('./HW2/data'), Path('/content/data'), Path('/content'),
    Path('/content/drive/MyDrive/HW2/data'),
]
if not any(all((p / name).is_file() for name in ['arithmetic_train.csv', 'arithmetic_eval.csv'])
           for p in candidates):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass
    candidates.append(Path('/content/drive/MyDrive/HW2/data'))
data_path = next((p for p in candidates if all((p / name).is_file() for name in
                 ['arithmetic_train.csv', 'arithmetic_eval.csv'])), None)
if data_path is None:
    raise FileNotFoundError('Upload both CSVs to data/ or set HW2_DATA_DIR to their remote directory.')
output_path = Path(os.environ.get('HW2_OUTPUT_DIR', str(data_path.parent if data_path.name == 'data' else data_path)))
output_path.mkdir(parents=True, exist_ok=True)
print('Data:', data_path.resolve(), '| Artifacts:', output_path.resolve())
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# Generative AI assistance: Codex helped implement the TODOs below.
# Stable IDs, constructed from training characters only (no eval leakage).
characters = sorted(set(''.join(df_train['src'])))
char_to_id = {token: index for index, token in enumerate(['<pad>', '<eos>'] + characters)}
id_to_char = {index: token for token, index in char_to_id.items()}
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# Teacher forcing: x contains the prompt and true answer, without EOS.
# At '=' predict the first answer character; at the final digit predict EOS.
# Experimental mode: reorder answer digits only; prompts are unchanged.
answer_order = globals().get('answer_order', 'normal')
if answer_order not in {'normal', 'least_significant_digit_first'}:
    raise ValueError('Unsupported answer order')
def reorder_answer_digits(answer, order='normal'):
    if order == 'normal':
        return answer
    sign = '-' if answer.startswith('-') else ''
    digits = answer[1:] if sign else answer
    # Preserve malformed model outputs as errors instead of repairing syntax.
    if not digits.isdigit():
        return answer
    return sign + digits[::-1]

pad_id = char_to_id['<pad>']
eos_id = char_to_id['<eos>']
assert vocab_size < 254, 'Compact arithmetic vocabulary must fit in uint8.'
max_answer_len = int(df_train['tgt'].str.len().max())
max_sequence_len = int(df_train['src'].str.len().max())
encoded_lengths = np.empty(len(df_train), dtype=np.int32)
encoded_x = np.full((len(df_train), max_sequence_len), pad_id, dtype=np.uint8)
encoded_y = np.full_like(encoded_x, pad_id)
for row_index, sequence in enumerate(df_train['src']):
    answer_start = sequence.index('=') + 1
    ordered = sequence[:answer_start] + reorder_answer_digits(sequence[answer_start:], answer_order)
    token_ids = np.fromiter((char_to_id[c] for c in ordered), dtype=np.uint8, count=len(ordered))
    length = len(token_ids)
    encoded_lengths[row_index] = length
    encoded_x[row_index, :length] = token_ids
    encoded_y[row_index, answer_start - 1:length - 1] = token_ids[answer_start:]
    encoded_y[row_index, length - 1] = eos_id
# Preserve the template Dataset/DataFrame interface with compact row indices.
# Raw strings and millions of Python token lists are no longer retained.
row_indices = np.arange(len(df_train), dtype=np.int32)
df_train = pd.DataFrame({'char_id_list': row_indices, 'label_id_list': row_indices})
data_storage = 'uint8 matrices; cast to torch.long per batch'
print('Compact token storage MiB:', (encoded_x.nbytes + encoded_y.nbytes + encoded_lengths.nbytes) / 2**20)
df_train.head()


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [ ]:
batch_size = 64
epochs = 40
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1
# Generative AI assistance: Codex suggested EM-driven continuation experiments.
# Keep the first continuation compatible with the existing 256-dim checkpoint.
# Later capacity experiments may change hidden_dim after inspecting the EM curve.
target_em = None


# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
        self.sequence_rows = sequences['char_id_list'].to_numpy(copy=False)
    
    def __len__(self):
        # return the amount of data
        return len(self.sequences)
    
    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        row_index = int(self.sequence_rows[index])
        length = int(encoded_lengths[row_index])
        x = encoded_x[row_index, :length]
        y = encoded_y[row_index, :length]
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0], dtype=torch.long) for data in batch]
    batch_y = [torch.tensor(data[1], dtype=torch.long) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(
    ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn,
    num_workers=0, pin_memory=torch.cuda.is_available(),
    generator=torch.Generator().manual_seed(2))

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        self.answer_order = globals().get('answer_order', 'normal')
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer
        
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        
        self.rnn_layer1 = torch.nn.GRU(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.GRU(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=200):
        
        char_list = [char_to_id[c] for c in start_char]
        
        next_char = None
        
        while len(char_list) < max_len: 
            # Cache the two LSTM states: consume the prompt once, then only
            # the previous generated character. No true answer is supplied.
            with torch.no_grad():
                first_step = len(char_list) == len(start_char)
                token_ids = char_list if first_step else [char_list[-1]]
                x = torch.tensor([token_ids], dtype=torch.long,
                                 device=next(self.parameters()).device)
                x = self.embedding(x)
                x, state1 = self.rnn_layer1(x, None if first_step else state1)
                x, state2 = self.rnn_layer2(x, None if first_step else state2)
                y = self.linear(x[:, -1, :])
                next_char = int(y.argmax(dim=-1).item())
            
            if next_char == char_to_id['<eos>']:
                break
            
            char_list.append(next_char)
            
        generated = ''.join(id_to_char[ch_id] for ch_id in char_list[len(start_char):])
        answer = reorder_answer_digits(generated, self.answer_order)
        return list(start_char + answer)

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('HW2 requires GPU training. Select a Colab GPU runtime before running this cell.')
torch.cuda.manual_seed_all(2)
print('Running environment: Colab GPU /', torch.cuda.get_device_name(0))
print('Python version:', platform.python_version())

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [ ]:
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
best_em = -1.0
history = []
# Record versions from the actual training kernel, rather than guessed pins.
package_names = ['numpy', 'pandas', 'torch', 'matplotlib', 'seaborn',
                 'scikit-learn', 'tqdm']
# Different distributions can provide the template's opencc import.
package_names += importlib.metadata.packages_distributions().get('opencc', [])
(output_path / 'requirements.txt').write_text(''.join(
    f'{name}=={importlib.metadata.version(name)}\n' for name in package_names), encoding='utf-8')
# Generative AI assistance: Codex assisted with resuming and learning-rate scheduling.
import json
import hashlib
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

deadline = None
def seconds_remaining(now=None):
    return float('inf')

def can_start_epoch(remaining, previous_duration=None):
    return True


config = dict(cell_type=cell_type, embed_dim=embed_dim, hidden_dim=hidden_dim, vocab_size=vocab_size,
              batch_size=batch_size, seed=2, optimizer='Adam',
              grad_clip=grad_clip, char_to_id=char_to_id, answer_order=answer_order)
order_suffix = '_lsf' if answer_order == 'least_significant_digit_first' else ''
run_path = output_path / 'state'
run_path.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_path / 'checkpoint.pt'
resume_path = run_path / 'resume.pt'
model = model.to(device)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=1, threshold=0.001, min_lr=1e-5)
start_epoch = 1
resume_batch = 0
loaded_checkpoint = False
previous_duration = None
stop_reason = None

if resume_path.is_file():
    # Keep uncapturable Adam step counters on CPU; load_state_dict moves moments
    # to the parameter device. Mapping the whole payload to MPS adds synchronizations.
    state = torch.load(resume_path, map_location='cpu', weights_only=True)
    state['config'].setdefault('answer_order', 'normal')
    state['config'].setdefault('cell_type', 'LSTM')
    if state['config'] != config:
        raise ValueError('Resume configuration differs. Use a separate run directory.')
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    history = state['history']
    best_em = state['best_em']
    start_epoch = state['next_epoch']
    resume_batch = state['next_batch']
    previous_duration = state.get('previous_duration')
    loaded_checkpoint = True
    del state
    print('Resuming:', resume_path, 'epoch:', start_epoch, 'batch:', resume_batch)
elif checkpoint_path.is_file():
    candidate = torch.load(checkpoint_path, map_location=device, weights_only=True)
    try:
        model.load_state_dict(candidate)
    except RuntimeError as exc:
        raise ValueError('Existing checkpoint has a different architecture. Preserve it and select matching dimensions before continuation.') from exc
    # Preserve the uploaded baseline before any new training.
    if not (run_path / 'initial_checkpoint.pt').exists():
        shutil.copy2(checkpoint_path, run_path / 'initial_checkpoint.pt')
    loaded_checkpoint = True
    print('Loaded existing checkpoint; optimizer starts fresh. Its full EM will be verified before training.')
else:
    print('No checkpoint found: starting a new model. Upload checkpoint.pt to', output_path, 'to continue the previous run.')

def save_resume(next_epoch, next_batch=0):
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                   scheduler=scheduler.state_dict(), config=config, history=history,
                   best_em=float(best_em), next_epoch=int(next_epoch),
                   next_batch=int(next_batch), previous_duration=previous_duration)
    temporary = resume_path.with_suffix('.tmp')
    torch.save(payload, temporary)
    temporary.replace(resume_path)

def save_best(score, epoch):
    # Keep state_dict as the exact submission interface, with metadata alongside.
    temporary = checkpoint_path.with_suffix('.tmp')
    torch.save(model.state_dict(), temporary)
    checkpoint_digest = hashlib.sha256(temporary.read_bytes()).hexdigest()
    temporary.replace(checkpoint_path)
    torch.save(model.state_dict(), run_path / 'best.pt')
    metadata = dict(config=config, epoch=int(epoch), exact_match=float(score), checkpoint_sha256=checkpoint_digest,
                    data_storage=globals().get('data_storage', 'Python lists'),
                    evaluated_examples=len(df_eval),
                    evaluation='Full CPU greedy generation with a copied model; GPU training, no teacher forcing at evaluation',
                    python=platform.python_version(), pytorch=str(torch.__version__),
                    training_progress=globals().get('checkpoint_training_progress'),
                    gpu=(torch.cuda.get_device_name(0) if device.type == 'cuda' else
                         'Apple GPU (MPS)' if device.type == 'mps' else 'CPU smoke test'),
                    device_type=device.type, system=platform.platform(),
                    timestamp=datetime.now(ZoneInfo('Asia/Taipei')).isoformat())
    metadata_tmp = output_path / 'checkpoint_metadata.tmp'
    metadata_tmp.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    metadata_tmp.replace(output_path / 'checkpoint_metadata.json')

def save_history():
    verify_protected_submission()
    (run_path / 'history.json').write_text(
        json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')

print('No time limit; no target-score stop.', flush=True)
print('Run artifacts:', run_path)

# A score threshold is disabled by default. Convergence is an operational
# criterion, not a proof of a mathematical global optimum.
def convergence_status(records, min_delta=0.0001, patience=12, low_lr=1e-5, min_low_lr_epochs=6):
    anchor = -1.0
    stagnant = 0
    low_epochs = 0
    for record in records:
        score = float(record['exact_match'])
        if score > anchor + min_delta:
            anchor = score
            stagnant = 0
        elif record.get('completed_epoch', True):
            stagnant += 1
        if not record.get('completed_epoch', True):
            continue
        used_lr = record.get('learning_rate', float('inf'))
        if used_lr <= low_lr and record.get('trained_batches_this_invocation', 1) > 0:
            low_epochs += 1
        else:
            low_epochs = 0
    return dict(converged=stagnant >= patience and low_epochs >= min_low_lr_epochs,
                meaningful_best=anchor, stagnant_epochs=stagnant, low_lr_epochs=low_epochs)

print('Convergence rule: 12 epochs without >0.01 percentage-point EM improvement,')
print('including at least 6 completed epochs trained at lr <= 1e-5. No deadline.')


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0

phase = 'before_training'
next_epoch_to_save = start_epoch
next_batch_to_save = resume_batch
optimizer_update_in_progress = False

try:
    # Verify an uploaded/resumed checkpoint using complete generated answers before
    # allowing any new model to replace it. Never use a partial validation score.
    if loaded_checkpoint and not history and seconds_remaining() > 90:
        phase = 'baseline_evaluation'
        model.eval()
        baseline_matched = 0
        baseline_total = 0
        for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc='Verify uploaded checkpoint'):
            if seconds_remaining() <= 10:
                break
            prediction = ''.join(model.generator(row['src'], max_len=len(row['src']) + max_answer_len + 1))
            baseline_matched += int(prediction[len(row['src']):] == row['tgt'])
            baseline_total += 1
        if baseline_total == len(df_eval):
            best_em = baseline_matched / baseline_total
            history.append(dict(epoch=0, exact_match=best_em, kind='verified uploaded checkpoint'))
            save_best(best_em, 0)
            save_history()
            print(f'Verified starting EM: {best_em:.6f}')
        else:
            stop_reason = 'deadline reached during checkpoint verification; existing weights preserved'

    if target_em is not None and best_em >= target_em:
        stop_reason = 'target reached by verified checkpoint'

    for epoch in __import__("itertools").count(start_epoch):
        if stop_reason is not None:
            break
        if not can_start_epoch(seconds_remaining(), previous_duration):
            stop_reason = 'insufficient time for another epoch before 22:30'
            break
        phase = 'training'
        next_epoch_to_save = epoch
        next_batch_to_save = resume_batch if epoch == start_epoch else 0
        # A durable boundary is available even if the first update is interrupted.
        save_resume(next_epoch_to_save, next_batch_to_save)
        epoch_start = time.monotonic()
        running_loss = 0.0
        supervised_tokens = 0
        processed_batches = 0
        interrupted = False
        partial_epoch = False
        # Repeatable order permits recovery at an exact batch boundary.
        dl_train.generator.manual_seed(2 + epoch)
        evaluation_only_resume = epoch == start_epoch and resume_batch >= len(dl_train)
        bar = [] if evaluation_only_resume else tqdm(dl_train, desc=f"Train epoch {epoch}")
        for batch_index, (batch_x, batch_y, batch_x_lens, batch_y_lens) in enumerate(bar):
            if epoch == start_epoch and batch_index < resume_batch:
                continue
            next_epoch_to_save = epoch
            next_batch_to_save = batch_index
            if ((output_path / 'STOP').exists() or (output_path.parent / 'STOP').exists()):
                save_resume(epoch, batch_index)
                stop_reason = 'manual STOP request; batch boundary saved'
                partial_epoch = True
                break
            if seconds_remaining() <= 120:
                save_resume(epoch, batch_index)
                stop_reason = 'deadline reserve reached; final partial epoch fully evaluated'
                partial_epoch = True
                break
            # Generative AI assistance: Codex helped implement training and EM.
            model.train()
            optimizer.zero_grad(set_to_none=True)

            batch_pred_y = model(batch_x.to(device), batch_x_lens)

            loss = criterion(batch_pred_y.reshape(-1, vocab_size),
                             batch_y.to(device).reshape(-1))
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite training loss; previous best checkpoint is preserved.')
            loss.backward()

            torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

            optimizer_update_in_progress = True
            optimizer.step()
            optimizer_update_in_progress = False
            next_batch_to_save = batch_index + 1
            token_count = int((batch_y != char_to_id['<pad>']).sum())
            running_loss += float(loss.item()) * token_count
            supervised_tokens += token_count
            processed_batches += 1
            i += 1
            if i % 50 == 0:
                bar.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'], best_em=best_em)
            if (batch_index + 1) % 1000 == 0:
                save_resume(epoch, batch_index + 1)
        if interrupted:
            break
        resume_batch = 0

        phase = 'evaluation'
        next_epoch_to_save = epoch
        if not partial_epoch:
            next_batch_to_save = len(dl_train)
        # Evaluate your model: full answers from generator(), never target prefixes.
        model.eval()
        evaluation_model = deepcopy(model).to('cpu').eval()
        matched = 0
        total = 0
        errors = []
        groups = {}
        evaluation_complete = True
        bar_eval = tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"Validation epoch {epoch}")
        for _, row in bar_eval:
            if seconds_remaining() <= 10:
                evaluation_complete = False
                break
            batch_x = row['src']
            batch_y = row['tgt']

            prediction = ''.join(evaluation_model.generator(batch_x, max_len=len(batch_x) + max_answer_len + 1))
            prediction = prediction[len(batch_x):]

            correct = int(prediction == batch_y)
            matched += correct
            total += 1
            categories = [f'answer_length_{len(batch_y)}',
                          'multiplication' if '*' in batch_x else 'addition_subtraction',
                          'parentheses' if '(' in batch_x else 'no_parentheses',
                          'negative_answer' if batch_y.startswith('-') else 'nonnegative_answer']
            for category in categories:
                counts = groups.setdefault(category, {'correct': 0, 'total': 0})
                counts['correct'] += correct
                counts['total'] += 1
            if not correct and len(errors) < 100:
                errors.append(dict(src=batch_x, target=batch_y, prediction=prediction))
        epoch_elapsed = time.monotonic() - epoch_start
        if not evaluation_only_resume and not partial_epoch:
            previous_duration = epoch_elapsed
        del evaluation_model
        if not evaluation_complete or total != len(df_eval):
            save_resume(epoch if partial_epoch else epoch + 1, next_batch_to_save if partial_epoch else 0)
            stop_reason = 'deadline reached during evaluation; partial EM is not used'
            break
        em = matched / total
        epoch_loss = running_loss / supervised_tokens if supervised_tokens else None
        checkpoint_training_progress = dict(epoch=epoch, completed_epoch=not partial_epoch,
                                            completed_batches=next_batch_to_save, total_batches=len(dl_train))
        record = dict(epoch=epoch, completed_epoch=not partial_epoch, exact_match=em, train_answer_token_loss=epoch_loss,
                      learning_rate=float(optimizer.param_groups[0]['lr']),
                      seconds=epoch_elapsed, evaluated_examples=total, evaluation_device='cpu',
                      trained_batches_this_invocation=processed_batches,
                      resumed_completed_training=evaluation_only_resume,
                      seconds_scope='partial epoch checkpoint' if partial_epoch else 'evaluation-only resume' if evaluation_only_resume else 'full epoch')
        history.append(record)
        if em > best_em:
            best_em = em
            save_best(em, epoch)
            print(f'Best checkpoint saved: epoch={epoch}, EM={em:.6f}')
        (run_path / f'errors_epoch_{epoch}.json').write_text(
            json.dumps(dict(groups=groups, examples=errors), ensure_ascii=False, indent=2), encoding='utf-8')
        if not partial_epoch:
            scheduler.step(em)
        save_history()
        save_resume(epoch if partial_epoch else epoch + 1, next_batch_to_save if partial_epoch else 0)
        next_epoch_to_save = epoch if partial_epoch else epoch + 1
        if not partial_epoch:
            next_batch_to_save = 0
        phase = 'between_epochs'
        print('EM history (copy into report):', history)
        print('Checkpoint:', checkpoint_path)
        print(matched/total)
        status = convergence_status(history)
        print('Convergence status:', status)
        if partial_epoch:
            break
        if status['converged']:
            stop_reason = 'operational convergence after low-learning-rate refinement'
            break
        if target_em is not None and em >= target_em:
            stop_reason = 'target full-eval EM reached'
            break
except KeyboardInterrupt:
    if phase == 'baseline_evaluation':
        stop_reason = 'checkpoint verification interrupted; existing weights preserved'
    elif optimizer_update_in_progress:
        # An optimizer may have updated only some tensors. Keep the previous
        # durable checkpoint rather than pretending this batch was completed.
        stop_reason = 'optimizer update interrupted; previous durable resume state retained'
    else:
        save_resume(next_epoch_to_save, next_batch_to_save)
        stop_reason = f'manual interruption during {phase}; completed boundary saved'
except Exception as exc:
    stop_reason = f'training error: {type(exc).__name__}: {exc}'
    raise
finally:
    outcome = dict(pid=os.getpid(), invocation_id=globals().get('invocation_id'), stop_reason=stop_reason or 'configured epochs completed', best_em=float(best_em),
                   completed_evaluation_epochs=max((row['epoch'] for row in history), default=0),
                   convergence=convergence_status(history), target_em=target_em,
                   deadline=None,
                   finished=datetime.now(ZoneInfo('Asia/Taipei')).isoformat())
    (run_path / 'outcome.json').write_text(json.dumps(outcome, indent=2), encoding='utf-8')
    print('Stop reason:', stop_reason or 'configured epochs completed')
    print('Best verified full-eval EM:', best_em)
    print('Best continuation weights:', checkpoint_path)
    print('Actual-runtime packages:', output_path / 'requirements.txt')
    print('Continuation state:', resume_path)
    # Leave the best model active in the kernel, rather than the last model.
    if checkpoint_path.is_file():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        model.eval()


# Research execution snapshot
Configuration and measured results are stored alongside this file; this is not the submitted homework report.
